# Random and diverse ensembles at fixed targeted SRO

This tutorial uses **m_scraps v0.6.0 or later**. From a source checkout, install with `python -m pip install ".[notebooks]"`, then select that Python environment in Jupyter. Native package compilation requires MPI and Eigen as described in the README; this notebook invokes no native optimizer.

We generate random and diversity-selected configurations of **(Ti₀.₅Zr₀.₅)₂(Al₀.₅Si₀.₅)C**, verify their targeted SRO, export structures, and inspect the **frozen 32-structure energy example**. No ATAT, icet, MACE, MaterialsFramework, checkpoint download, or energy calculation is required. The small generation runs are capped; this is a tutorial, not a benchmark campaign.

The goal is configuration-dependent variability at prescribed SRO. Equilibrium sampling and phase stability are outside this workflow.

In [ ]:
import hashlib
import inspect
import json
from pathlib import Path
from tempfile import mkdtemp

import matplotlib.pyplot as plt
import numpy as np
from ase import Atoms
from ase.io import read, write
from IPython.display import Markdown, display
import m_scraps
from m_scraps import generate_ensemble, analyze_structure
from atat_reference import figure_model  # Bundled FIGURES reader; does not run ATAT.

fixture = Path(inspect.getfile(generate_ensemble)).parent / "fixtures/max_ensemble"
assert fixture.is_dir(), "Install m_scraps v0.6.0 or later with its bundled fixtures."
work = Path(mkdtemp(prefix="m_scraps_ensemble_tutorial_"))
print("m_scraps:", m_scraps.__version__)
print("Outputs:", work)

def table(headers, rows):
    display(Markdown("| " + " | ".join(headers) + " |\n| " + " | ".join(["---"] * len(headers)) + " |\n" +
                     "\n".join("| " + " | ".join(map(str, row)) + " |" for row in rows)))

## 1. Load a common parent cell and explicit sublattices

The 2 × 2 × 1 MAX211 supercell has **32 sites**: Ti₈Zr₈Al₄Si₄C₈. Ti/Zr mix on M sites, Al/Si on A sites, and carbon remains fixed. The prescribed lattice is illustrative, not an optimized alloy geometry.

We use the bundled parent coordinates and pair/triplet FIGURES. Reconstructing the coordinates from FIGURES preserves their exact site order and avoids rounding at periodic boundaries in text structure files.

In [ ]:
saved = json.loads((fixture / "random32/result.json").read_text())
figures = fixture / "random32/FIGURES"
model = figure_model(figures)
initial = Atoms(read(fixture / "initial.extxyz").get_chemical_symbols(),
                scaled_positions=model["positions"], cell=model["cell"], pbc=True)
config, targets = saved["sublattices"], saved["targets"]
assert len(initial) == 32
print(initial.get_chemical_formula())
print("Cell lengths (Å):", initial.cell.lengths())
table(["Source", "Destination", "Pair", "Shell", "Target α"],
      [[t["source"], t["destination"], t["center_species"] + "–" + t["neighbor_species"],
        t["shell"], t["target"]] for t in targets])

## 2. Generate two equally sized ensembles

Both modes use identical SA settings, composition constraints and an **exact zero-error tolerance** for the three targeted first-shell SRO components. Pair/triplet correlations serve as diversity descriptors, including components not constrained by those targets.

- **Random:** sample without replacement from the collected symmetry-distinct pool. The selection seed does not change candidate generation.
- **Diversity:** start with the lowest-error candidate and greedily spread selections in correlation space. Energy is never used in selection.

We request 32 structures from a pool capped at 128. The time cap is cooperative. If discovery stops early, inspect the stopping reason and increase the budget deliberately; do not silently weaken the SRO tolerance.

In [ ]:
options = dict(cutoff=3.5, shells=2, targets=targets, sublattices=config,
               error_tolerance=0, count=32, method="sa", starts=8, steps=6000,
               seed=42, max_pool=128, max_seconds=60, figures=figures,
               species=saved["diversity"]["species"])
reports, structures = {}, {}
for mode in ("random", "diversity"):
    reports[mode], structures[mode] = generate_ensemble(
        initial, **options, selection=mode, selection_seed=17)
    print(mode, reports[mode]["selected_count"], "selected;", reports[mode]["stop_reason"])
    assert reports[mode]["selected_count"] == 32, "Insufficient candidates: inspect budgets before continuing."
same_pool = (reports["random"]["selection"]["pool_occupations_sha256"] ==
             reports["diversity"]["selection"]["pool_occupations_sha256"])
assert same_pool, "Different pools: a timing limit may have interrupted discovery; do not compare selectors yet."
print("Identical ordered candidate pools:", same_pool)

## 3. Verify constraints and compare correlation coverage

Equal targeted SRO does not imply identical local environments. Untargeted pairs and triplets can differ. RMS descriptor distances use equal component weights without variance rescaling; they are not energy distances. Distinct structures may have identical descriptors.

In [ ]:
comparison = []
for mode, ensemble in structures.items():
    report = reports[mode]
    hashes = [e["occupations_sha256"] for e in report["selected"]]
    assert len(set(hashes)) == len(ensemble)
    for atoms in ensemble:
        np.testing.assert_allclose(atoms.cell, initial.cell)
        analysis = analyze_structure(atoms, cutoff=3.5, shells=2,
                                     sublattices=config, sro_targets=targets)
        assert analysis["sro_comparison"]["metrics"]["max_error"] == 0
        for group, composition in analysis["composition"].items():
            assert composition["counts"] == config["counts"][group]
    matrix = np.array(report["diversity"]["spread"]["pairwise_rms_distances"])
    distances = matrix[np.triu_indices(len(ensemble), 1)]
    comparison.append([mode, len(ensemble), report["search"]["pool_size"], 0,
                       f"{distances.mean():.4f}", f"{distances.min():.4f}"])
table(["Selection", "Structures", "Pool size", "Max targeted error", "Mean pairwise RMS", "Minimum RMS"], comparison)

## 4. Export structures and provenance

Each mode gets its own directory. The JSON records the candidate-pool hashes, selection seed, exact SRO errors, descriptor vectors and source/dependency provenance. Keep these with any subsequent property calculations.

In [ ]:
for mode, ensemble in structures.items():
    out = work / mode
    out.mkdir()
    (out / "result.json").write_text(json.dumps(reports[mode], indent=2) + "\n")
    for entry, atoms in zip(reports[mode]["selected"], ensemble):
        directory = out / entry["id"]
        directory.mkdir()
        write(directory / "structure.extxyz", atoms)
        write(directory / "structure.cif", atoms)
print("Exported both ensembles to:", work)

## 5. Inspect archived energies without an ML installation

The bundled energy cohort contains **32 random structures selected from 128 candidates**, with selection seed 17. MACE predictions were computed externally through MaterialsFramework using CPU float64 and fixed-cell FIRE relaxation (force tolerance 0.03 eV/Å; at most 300 steps). All 32 converged and retained parent-site correspondence.

The following values belong to the **archived structures**, not automatically to the configurations just generated. We report whether the newly selected occupation hashes match; floating-point, dependency or time-budget differences can change a fresh search. No diversity-versus-random energy comparison is inferred here.

In [ ]:
checksums = json.loads((fixture / "checksums.json").read_text())["files"]
for name, expected in checksums.items():
    assert hashlib.sha256((fixture / name).read_bytes()).hexdigest() == expected, name
energy_results = json.loads((fixture / "random32-energies/summary.json").read_text())
manifest = json.loads((fixture / "random32-energies/manifest.json").read_text())
rows = energy_results["structures"]
assert len(rows) == 32
accepted = [r for r in rows if r["force_converged"] and r["site_mapping_valid"]]
assert len(accepted) == 32
assert all(r["initial_sro_metrics"]["max_error"] == 0 for r in rows)
fresh_hashes = [e["occupations_sha256"] for e in reports["random"]["selected"]]
archived_hashes = [e["occupations_sha256"] for e in saved["selected"]]
print("Fresh random occupations match archive:", fresh_hashes == archived_hashes)
print("Archived model SHA-256:", manifest["model_sha256"])
print("Frozen raw energy CSV:", fixture / "random32-energies/energies.csv")

In [ ]:
energies = {
    "Unrelaxed": np.array([r["unrelaxed_eV_per_atom"] for r in accepted]),
    "Fixed-cell relaxed": np.array([r["relaxed_eV_per_atom"] for r in accepted]),
}
frozen_stats = json.loads((fixture / "distribution-summary.json").read_text())["stats"]
summary_rows = []
for name, values in energies.items():
    sd, span = 1000 * values.std(), 1000 * np.ptp(values)
    np.testing.assert_allclose(sd, frozen_stats[name]["population_std_meV_per_atom"], atol=1e-10)
    summary_rows.append([name, len(values), f"{values.mean():.6f}", f"{sd:.3f}", f"{span:.3f}"])
table(["Stage", "N", "Mean (eV/atom)", "Population SD (meV/atom)", "Range (meV/atom)"], summary_rows)

fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
for ax, (name, values) in zip(axes, energies.items()):
    centered = 1000 * (values - values.mean())
    ax.hist(centered, bins=8, color="#267c98", edgecolor="white")
    ax.plot(centered, np.full(len(values), -.12), "|", color="black", clip_on=False)
    ax.axvline(0, color="#bb552e", linestyle="--", linewidth=1)
    ax.set(title=name, xlabel="Energy relative to sample mean (meV/atom)", ylabel="Structure count")
fig.suptitle("Archived random ensemble: energy variability at fixed targeted SRO")
fig.savefig(work / "archived-energy-variability.png", dpi=150)
plt.show()

## Interpretation and next use

The archived energy spread is **configuration-dependent variability at fixed targeted SRO**. The histogram is conditional on a bounded SA-generated pool; it is not a converged equilibrium distribution, a free energy, or a thermodynamic uncertainty interval. Equal initial occupation-based SRO does not establish identical distance-based SRO after atomic relaxation. The potential and prescribed cell also require validation before making material-specific predictions.

Use **diversity selection** to cover different descriptor environments with a limited property-evaluation budget. Use **random selection** for reproducible subsets of the available candidate pool. Neither method provides Boltzmann sampling or orbit-multiplicity weighting.

You can change the selection seed, repeat the comparison, or export structures to an external property workflow. Keep the candidate-pool hashes fixed when comparing selectors. The notebook ends here; it does not download a model or launch an energy campaign.